# 日本語 BERT の学習を GPU メモリに収める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/08_transformers/bert_training_memory.ipynb)

NLP クックブック 08-14 ／ Transformers の基本操作

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

日本語の口コミ8件で分類器を1回更新するのに、何秒と何 MiB かかるだろうか。入力と初期重みをそろえ、バッチの分割、勾配チェックポイント、系列長の効果を順に測る。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import gc
import time
from statistics import median

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed

if not torch.cuda.is_available():
    raise RuntimeError("Colab のランタイムを GPU に変更してください")
print(torch.cuda.get_device_name())
name = "tohoku-nlp/bert-base-japanese-char-v2"
tok = AutoTokenizer.from_pretrained(name)
reviews = ["操作が簡単で助かった。", "説明が少なく迷った。", "届くのが早かった。", "箱が壊れていた。",
           "音が静かで快適だった。", "動作が遅くて困った。", "色も形も気に入った。", "すぐ電池が切れた。"]
# 長さの異なる教材用の入力を作る。偶数番が好意的、奇数番が否定的。
texts = [s * (i + 1) for i, s in enumerate(reviews)]
labels = torch.tensor([1, 0] * 4, device="cuda")
lengths = [len(ids) for ids in tok(texts)["input_ids"]]

Tesla T4


In [3]:
def measure(batch_size, checkpoint, max_length):
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2).cuda().train()
    if checkpoint:
        model.gradient_checkpointing_enable()
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
    batch = tok(texts, truncation=True, padding="max_length", max_length=max_length,
                return_tensors="pt").to("cuda")
    accumulation = len(texts) // batch_size

    def update():
        opt.zero_grad(set_to_none=True)
        for start in range(0, len(texts), batch_size):
            part = slice(start, start + batch_size)
            inputs = {k: v[part] for k, v in batch.items()}
            loss = model(**inputs, labels=labels[part]).loss
            (loss / accumulation).backward()
        opt.step()

    update()  # Adam の状態と CUDA カーネルを準備する
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    seconds = []
    for _ in range(3):
        start = time.perf_counter()
        update()
        torch.cuda.synchronize()
        seconds.append(time.perf_counter() - start)
    return {"バッチ": batch_size, "累積回数": accumulation, "チェックポイント": checkpoint,
            "系列長": max_length, "切り詰め件数": sum(n > max_length for n in lengths),
            "ピーク MiB": round(torch.cuda.max_memory_allocated() / 2**20),
            "秒/更新（中央値）": round(median(seconds), 3)}

rows = []
for setting in [(8, False, 128), (2, False, 128), (2, True, 128), (2, True, 32)]:
    gc.collect()
    torch.cuda.empty_cache()
    rows.append(measure(*setting))
print(pd.DataFrame(rows).to_string(index=False))

 バッチ  累積回数  チェックポイント  系列長  切り詰め件数  ピーク MiB  秒/更新（中央値）
   8     1     False  128       0     1776      0.170
   2     4     False  128       0     1756      0.218
   2     4      True  128       0     1750      0.299
   2     4      True   32       5     1750      0.247


- 1行目と2行目はどちらも8件で1回更新する。勾配累積では小分けの損失を累積回数で割る。バッチを分けると、同時に保持する中間結果が減る。
- 2行目と3行目で、中間結果を逆伝播時に再計算するコストを読む。3行目と4行目では、節約できたメモリと、末尾が切れた口コミの件数を合わせて見る。
- ピークはモデル、勾配、Adam の状態を含む PyTorch の割り当て量。計測は同じ8件を繰り返す教材用の設定で、分類精度を評価するには別の検証データが要る。